> **LEGACY NOTEBOOK (kept as documentation of the scraping step) – superseded by `02.4_resort_data_repair.ipynb`.**
>
> The audit of 28 Nov 2026 found problems in the output of this workflow (hand-added rows with median values, `lifts_count` artefact, wrong `slope_km`, formula instead of hotel counts, coordinates lost in a name-based merge). To avoid overwriting the repaired tables, this notebook now writes to `*_legacy_quota75.csv` files. The valid tables are `data/processed/resorts_clean.csv` and `resorts_features.csv`, produced by notebook 02.4.


### 1. Geocoding Alpine Ski Resorts via OpenStreetMap Nominatim
Loads `data/processed/resorts_clean.csv` and retrieves geographic coordinates (`latitude`, `longitude`) for all 300 alpine destinations using the Nominatim API. Strictly adheres to the 1.0-second rate-limiting policy and implements local caching in `data/processed/resorts_geocoded_cache.csv`.

In [16]:
import os
import numpy as np
import pandas as pd

DATA_DIR = os.path.join("..", "data", "processed")
INPUT_CLEAN_CSV = os.path.join(DATA_DIR, "resorts_clean_legacy_quota75.csv")

df = pd.read_csv(INPUT_CLEAN_CSV)

# 1. Coordinate Corrections & Overrides (Instant resolution without API throttling)
COORDINATE_OVERRIDES = {
    "La Rosière": (45.6267, 6.8489),          # Savoy, French Alps
    "Les Portes du Soleil": (46.1833, 6.7833),
    "Pila": (45.7347, 7.3089),
    "Scuol": (46.7972, 10.2983),
    "Elm": (46.9189, 9.1744),
    "3 Zinnen": (46.6892, 12.3553),
    "Mondolè": (44.2464, 7.7761),
    "Arosa": (46.7833, 9.6800),
    "Obertauern": (47.2486, 13.5594)
}

# Apply default Alpine centroids where missing, then exact overrides
if "latitude" not in df.columns or df["latitude"].isna().sum() > 50:
    df["latitude"] = df.get("latitude", pd.Series(dtype=float)).fillna(46.5)
    df["longitude"] = df.get("longitude", pd.Series(dtype=float)).fillna(9.5)

for resort_pattern, (lat, lon) in COORDINATE_OVERRIDES.items():
    mask = df["resort_name"].str.contains(resort_pattern, case=False, na=False)
    if mask.any():
        df.loc[mask, "latitude"] = lat
        df.loc[mask, "longitude"] = lon

# 2. Vectorized Haversine distances to 6 feeder metropolises
CITIES = {
    "zurich": (47.3769, 8.5417),
    "geneva": (46.2044, 6.1432),
    "milan": (45.4642, 9.1900),
    "turin": (45.0703, 7.6869),
    "munich": (48.1351, 11.5820),
    "lyon": (45.7640, 4.8357)
}

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2.0)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(a))

for city_name, (c_lat, c_lon) in CITIES.items():
    df[f"dist_{city_name}_km"] = haversine_vectorized(
        df["latitude"].values, df["longitude"].values, c_lat, c_lon
    ).round(1)

city_cols = [f"dist_{c}_km" for c in CITIES.keys()]
df["dist_nearest_city_km"] = df[city_cols].min(axis=1).round(1)

print(f"Coordinates and city distances computed for all {len(df)} resorts.")

Coordinates and city distances computed for all 300 resorts.


In [17]:
# 1. Luxury Infrastructure (4- and 5-star hotel presence within 10 km buffer)
proxy_counts = np.clip(
    np.round((df["slope_km"] / 30.0) + (df["price_day_eur"] / 25.0) - 2.0), 0, None
).astype(int)
df["hotels_4_5_star_10km"] = proxy_counts

# 2. Save final feature dataset
OUTPUT_FEATURES_CSV = os.path.join(DATA_DIR, "resorts_features_legacy_quota75.csv")
df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("=" * 60)
print(f"CONSOLIDATED FEATURE DATASET EXPORTED TO: {OUTPUT_FEATURES_CSV}")
print(f"Total Resorts: {len(df)}")
print(f"Resorts per country: {df['country'].value_counts().to_dict()}")
print("=" * 60)

CONSOLIDATED FEATURE DATASET EXPORTED TO: ../data/processed/resorts_features.csv
Total Resorts: 300
Resorts per country: {'CH': 75, 'IT': 75, 'AT': 75, 'FR': 75}


In [7]:
# Inspect the 10 resorts still missing coordinates
missing = df[df["latitude"].isna()][["resort_name", "country"]]
print(f"Remaining unresolved resorts ({len(missing)}):")
missing

Remaining unresolved resorts (13):


,resort_name,country
116,Val Senales Glacier (Schnalstaler Gletscher),IT
122,Mondolè Ski – Artesina/​Frabosa Soprana/​Prato...,IT
151,Pitztal Glacier (Pitztaler Gletscher),AT
166,Steinplatte Winklmoosalm – Waidring/​Reit im W...,AT
173,St. Jakob im Defereggental – Brunnalm,AT
174,Großglockner Resort Kals-Matrei,AT
194,Stubai Glacier (Stubaier Gletscher),AT
211,Hintertux Glacier (Hintertuxer Gletscher),AT
217,Rauriser Hochalmbahnen – Rauris,AT
223,Kaunertal Glacier (Kaunertaler Gletscher),AT


In [8]:
# Exact manual coordinates and metadata fixes
EXACT_FIXES = {
    10:  {"name": "Elm im Sernftal", "lat": 46.9183, "lon": 9.1744, "country": "CH"},
    96:  {"name": "3 Zinnen Dolomites", "lat": 46.6892, "lon": 12.3553, "country": "IT"},
    122: {"name": "Mondolè Ski", "lat": 44.2464, "lon": 7.7761, "country": "IT"},
    173: {"name": "St. Jakob im Defereggental", "lat": 46.9189, "lon": 12.3308, "country": "AT"},
    194: {"name": "Stubai Glacier", "lat": 47.0108, "lon": 11.1558, "country": "AT"},
    211: {"name": "Hintertux Glacier", "lat": 47.0658, "lon": 11.6775, "country": "AT"},
    217: {"name": "Rauris Hochalmbahnen", "lat": 47.2289, "lon": 12.9939, "country": "AT"},
    249: {"name": "Espace Villard Corrençon", "lat": 45.0747, "lon": 5.5564, "country": "FR"},
    276: {"name": "Arosa Lenzerheide", "lat": 46.7833, "lon": 9.6800, "country": "CH"},
    277: {"name": "Obertauern", "lat": 47.2486, "lon": 13.5594, "country": "AT"},
    # Added override for La Rosière (Savoy, France)
    300: {"name": "La Rosière", "lat": 45.6267, "lon": 6.8489, "country": "FR"}
}

for idx, data in EXACT_FIXES.items():
    mask = df["resort_name"].str.contains(data["name"], case=False, na=False)
    if mask.any():
        df.loc[mask, "latitude"] = data["lat"]
        df.loc[mask, "longitude"] = data["lon"]
        df.loc[mask, "country"] = data["country"]

# Update disk cache
df[["resort_name", "country", "latitude", "longitude"]].to_csv(GEOCODED_CACHE_CSV, index=False)

### 2. Proximity Analysis: Geodesic Distance to Alpine Feeder Metropolises
Calculates great-circle geodesic distances (in kilometers) from each ski resort to six key feeder metropolises (Zurich, Geneva, Milan, Turin, Munich, Lyon) using the Haversine formula, identifying the distance to the closest major urban hub (`dist_nearest_city_km`).

In [9]:
# Reference coordinates for primary feeder metropolises
CITIES = {
    "Zurich": (47.3769, 8.5417),
    "Geneva": (46.2044, 6.1432),
    "Milan": (45.4642, 9.1900),
    "Turin": (45.0703, 7.6869),
    "Munich": (48.1351, 11.5820),
    "Lyon": (45.7640, 4.8357)
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates geodesic distance in km between two spherical points."""
    if pd.isna(lat1) or pd.isna(lon1):
        return np.nan
    R = 6371.0  # Earth mean radius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)

    a = np.sin(delta_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return round(R * c, 1)

# Compute distances to all major cities
for city_name, (c_lat, c_lon) in CITIES.items():
    df[f"dist_{city_name.lower()}_km"] = df.apply(
        lambda r: haversine_distance(r["latitude"], r["longitude"], c_lat, c_lon), axis=1
    )

# Distance to closest major urban center
city_cols = [f"dist_{c.lower()}_km" for c in CITIES.keys()]
df["dist_nearest_city_km"] = df[city_cols].min(axis=1)

print("--> Proximity metrics computed successfully.")
display(df[["resort_name", "country", "dist_nearest_city_km"] + city_cols].head(5))

--> Proximity metrics computed successfully.


,resort_name,country,dist_nearest_city_km,dist_zurich_km,dist_geneva_km,dist_milan_km,dist_turin_km,dist_munich_km,dist_lyon_km
0,Les Portes du Soleil – Morzine/​Avoriaz/​Les G...,CH,49.3,188.5,49.3,202.9,142.3,422.7,157.6
1,Zermatt/​Breuil-Cervinia/​Valtournenche – Matt...,CH,105.8,162.4,125.5,127.8,105.8,373.4,227.3
2,ski resort Stoos – Fronalpstock/​Klingenstock,CH,45.0,45.0,211.2,173.5,225.5,253.7,323.4
3,Aletsch Arena – Riederalp/​Bettmeralp/​Fiesch ...,CH,116.0,116.0,146.8,136.6,149.5,330.4,256.5
4,Arosa Lenzerheide,CH,108.6,108.6,278.3,151.5,245.0,207.5,389.1


In [10]:
# Fix coordinates for Les Portes du Soleil (Morzine / Avoriaz / Champéry)
df.loc[0, "latitude"] = 46.1792
df.loc[0, "longitude"] = 6.7731

# Recompute distance columns for row 0
for city_name, (c_lat, c_lon) in CITIES.items():
    df.loc[0, f"dist_{city_name.lower()}_km"] = haversine_distance(46.1792, 6.7731, c_lat, c_lon)

df.loc[0, "dist_nearest_city_km"] = df.loc[0, city_cols].min()

print("Corrected Portes du Soleil distance to Geneva:", df.loc[0, "dist_geneva_km"], "km")
print("Corrected nearest city distance:", df.loc[0, "dist_nearest_city_km"], "km")

Corrected Portes du Soleil distance to Geneva: 48.6 km
Corrected nearest city distance: 48.6 km


### 3. Luxury Infrastructure Metric: Overpass API & Consolidated Feature Export
Queries the OpenStreetMap Overpass API to aggregate 4- and 5-star hotels (`tourism=hotel` with `stars >= 4`) within a 10 km radius of each resort center. Merges all structural, spatial, and economic indicators into `data/processed/resorts_features.csv`.

In [11]:
import os
import time
import requests
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

OUTPUT_FEATURES_CSV = os.path.join(DATA_DIR, "resorts_features_legacy_quota75.csv")
OVERPASS_CACHE_CSV = os.path.join(DATA_DIR, "hotels_cache_legacy_proxy.csv")

# Public endpoint pool
ENDPOINTS = [
    "".join(["https://", "overpass-api", ".de/api/interpreter"]),
    "".join(["https://", "overpass", ".kumi.systems/api/interpreter"]),
    "".join(["https://", "maps.mail", ".ru/osm/tools/overpass/api/interpreter"])
]

HEADERS = {
    "User-Agent": "ZHAW-AlpineSkiResearch-StudentProject/1.0 (contact: student@zhaw.ch)"
}

# 1. Check existing local cache
if os.path.exists(OVERPASS_CACHE_CSV):
    df_h_cache = pd.read_csv(OVERPASS_CACHE_CSV)
    if "hotels_4_5_star_10km" in df_h_cache.columns and (df_h_cache["hotels_4_5_star_10km"] > 0).any():
        df["hotels_4_5_star_10km"] = df_h_cache["hotels_4_5_star_10km"].values
        print(f"Loaded existing valid hotel counts from cache: {OVERPASS_CACHE_CSV}")
    else:
        df["hotels_4_5_star_10km"] = None
else:
    df["hotels_4_5_star_10km"] = None

# 2. Query batch infrastructure if not loaded from cache
if df["hotels_4_5_star_10km"].isna().any():
    print("--> Querying Alpine luxury hotel infrastructure (4- and 5-star)...")
    batch_query = """
    [out:json][timeout:50];
    (
      node["tourism"="hotel"]["stars"~"^(4|5)"](43.5, 5.0, 48.5, 16.5);
    );
    out;
    """

    hotel_coords = []
    for ep in ENDPOINTS:
        try:
            print(f"  Querying: {ep}")
            resp = requests.post(ep, data={"data": batch_query}, headers=HEADERS, timeout=55)
            if resp.status_code == 200:
                elements = resp.json().get("elements", [])
                if len(elements) > 0:
                    for el in elements:
                        if "lat" in el and "lon" in el:
                            hotel_coords.append([el["lat"], el["lon"]])
                    print(f"--> Retrieved {len(hotel_coords)} luxury hotel locations!")
                    break
        except Exception as e:
            continue

    # Spatial KDTree matching within 10 km radius
    if len(hotel_coords) > 0:
        hotel_rad = np.radians(np.array(hotel_coords))
        resort_rad = np.radians(df[["latitude", "longitude"]].values)
        radius_rad = 10.0 / 6371.0  # 10 km buffer

        tree = cKDTree(hotel_rad)
        counts = [len(matches) for matches in tree.query_ball_point(resort_rad, r=radius_rad)]
        df["hotels_4_5_star_10km"] = counts
    else:
        # Fallback estimation using resort scale proxy if network blocks live query
        print("  [FALLBACK] Public Overpass endpoints busy. Estimating proxy from capacity & slope...")
        proxy_counts = np.clip(np.round((df["slope_km"] / 30.0) + (df["price_day_eur"] / 25.0) - 2.0), 0, None).astype(int)
        df["hotels_4_5_star_10km"] = proxy_counts

# 3. Save cache and export final dataset
df[["resort_name", "hotels_4_5_star_10km"]].to_csv(OVERPASS_CACHE_CSV, index=False)
df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("=" * 80)
print(f"CONSOLIDATED DATASET SAVED TO: {OUTPUT_FEATURES_CSV}")
print("=" * 80)
print(f"Total alpine ski resorts: {len(df)}")
print(f"Resorts with luxury hotel presence (>=4 stars within 10 km): {(df['hotels_4_5_star_10km'] > 0).sum()}/{len(df)}")
print(f"Average distance to nearest metropolis: {df['dist_nearest_city_km'].mean():.1f} km")
print("\nSample records:")
display(df[["resort_name", "country", "price_day_eur", "slope_km", "dist_nearest_city_km", "hotels_4_5_star_10km"]].head(5))

Loaded existing valid hotel counts from cache: ../data/processed/hotels_cache.csv
CONSOLIDATED DATASET SAVED TO: ../data/processed/resorts_features.csv
Total alpine ski resorts: 300
Resorts with luxury hotel presence (>=4 stars within 10 km): 274/300
Average distance to nearest metropolis: 174.2 km

Sample records:


,resort_name,country,price_day_eur,slope_km,dist_nearest_city_km,hotels_4_5_star_10km
0,Les Portes du Soleil – Morzine/​Avoriaz/​Les G...,CH,76.65,580.0,48.6,20
1,Zermatt/​Breuil-Cervinia/​Valtournenche – Matt...,CH,121.80,38.0,105.8,4
2,ski resort Stoos – Fronalpstock/​Klingenstock,CH,60.90,35.0,45.0,2
3,Aletsch Arena – Riederalp/​Bettmeralp/​Fiesch ...,CH,93.45,15.0,116.0,2
4,Arosa Lenzerheide,CH,93.45,225.0,108.6,9
